# Trabajo Práctico de Laboratorio N° 2

## Analizador Sintáctico Descendente Predictivo por Tabla

---
# 1. Verificación y modificación de la gramática

## 1.1. Gramática original

La gramática original de Mini-C proporcionada en el enunciado es:

```text
Program → DeclList

DeclList → DeclList Decl
         | Decl

Decl → FuncDecl
     | VarDecl

VarDecl → Type id ;
        | Type id = Expr ;

FuncDecl → Type id ( ParamList ) Block
         | Type id ( ) Block

Type → int | float | bool | void | string

ParamList → ParamList , Param
          | Param

Param → Type id

Block → { StmtList }

StmtList → StmtList Stmt
         | Stmt
         | λ

Stmt → VarDecl
     | id = Expr ;
     | id ( ArgList ) ;
     | id ( ) ;
     | IfStmt
     | WhileStmt
     | ForStmt
     | return Expr ;
     | return ;
     | print ( Expr ) ;
     | read ( id ) ;

IfStmt → if ( Expr ) Block else Block
       | if ( Expr ) Block

WhileStmt → while ( Expr ) Block

ForStmt → for ( id = Expr ; Expr ; id = Expr ) Block

Expr → Expr || RelExpr
     | Expr && RelExpr
     | RelExpr

RelExpr → AddExpr RelOp AddExpr
        | AddExpr

RelOp → == | != | < | > | <= | >=

AddExpr → AddExpr + Term
        | AddExpr - Term
        | Term

Term → Term * Unary
     | Term / Unary
     | Unary

Unary → ! Unary
      | - Unary
      | Factor

Factor → ( Expr )
        | id ( ArgList )
        | id ( )
        | id
        | num
        | str
        | true
        | false

ArgList → ArgList , Expr
        | Expr

### (El símbolo distinguido de la gramática es: Program)

## 1.2. Eliminación de recursividad izquierda

Se identificaron las siguientes producciones con recursividad izquierda directa:

* DeclList
* ParamList
* StmtList
* Expr
* AddExpr
* Term
* ArgList

Se aplicó el procedimiento de eliminación de recursividad izquierda, obteniendo:

-> DeclList
DeclList → Decl DeclList'
DeclList' → Decl DeclList'
          | λ

->ParamList
ParamList → Param ParamList'
ParamList' → , Param ParamList'
           | λ

->StmtList
StmtList → Stmt StmtList'
StmtList' → Stmt StmtList'
          | λ

->Expr
Expr → RelExpr Expr'
Expr' → || RelExpr Expr'
      | && RelExpr Expr'
      | λ

->AddExpr
AddExpr → Term AddExpr'
AddExpr' → + Term AddExpr'
         | - Term AddExpr'
         | λ

->Term
Term → Unary Term'
Term' → * Unary Term'
      | / Unary Term'
      | λ

->ArgList
ArgList → Expr ArgList'
ArgList' → , Expr ArgList'
         | λ

## 1.3. Factorizacion
Luego de eliminar la recursividad izquierda, se realizó la factorización de las producciones que presentan prefijos comunes.

->VarDecl
VarDecl → Type id A
A → ;
  | = Expr ;

->FuncDecl
FuncDecl → Type id ( B
B → ParamList ) Block
  | ) Block

->Decl
Decl → Type id C
C → ( B
  | A

->Stmt
Stmt → VarDecl
     | id I
     | IfStmt
     | WhileStmt
     | ForStmt
     | return J
     | print ( Expr ) ;
     | read ( id ) ;
I → = Expr ;
  | ( K
K → ArgList ) ;
  | ) ;
J → Expr ;
  | ;

->IfStmt
  IfStmt → if ( Expr ) Block D
D → else Block
  | λ

->RealExpr
RelExpr → AddExpr E
E → RelOp AddExpr
  | λ

->RelOp
RelOp → ==
      | !=
      | < F
      | > F
F → =
  | λ

->Factor
Factor → ( Expr )
       | id G
       | num
       | str
       | true
       | false
G → ( H
  | λ
H → ArgList )
  | )

## 1.4. Gramática modificada
Despues de eliminar la recursividad izquierda y realizar las factorizaciones necesarias, la gramática utilizada para el análisis sintáctico queda:

Program → DeclList

DeclList → Decl DeclList'

DeclList' → Decl DeclList'
          | λ

Decl → Type id C

C → ( B
  | A

A → ;
  | = Expr ;

VarDecl → Type id A

FuncDecl → Type id ( B

B → ParamList ) Block
  | ) Block

Type → int | float | bool | void | string

ParamList → Param ParamList'

ParamList' → , Param ParamList'
           | λ

Param → Type id

Block → { StmtList }

StmtList → Stmt StmtList'

StmtList' → Stmt StmtList'
          | λ

Stmt → VarDecl
     | id I
     | IfStmt
     | WhileStmt
     | ForStmt
     | return J
     | print ( Expr ) ;
     | read ( id ) ;

I → = Expr ;
  | ( K

K → ArgList ) ;
  | ) ;

J → Expr ;
  | ;

IfStmt → if ( Expr ) Block D

D → else Block
  | λ

WhileStmt → while ( Expr ) Block

ForStmt → for ( id = Expr ; Expr ; id = Expr ) Block

Expr → RelExpr Expr'

Expr' → || RelExpr Expr'
      | && RelExpr Expr'
      | λ

RelExpr → AddExpr E

E → RelOp AddExpr
  | λ

RelOp → ==
      | !=
      | < F
      | > F

F → =
  | λ

AddExpr → Term AddExpr'

AddExpr' → + Term AddExpr'
         | - Term AddExpr'
         | λ

Term → Unary Term'

Term' → * Unary Term'
      | / Unary Term'
      | λ

Unary → ! Unary
      | - Unary
      | Factor

Factor → ( Expr )
       | id G
       | num
       | str
       | true
       | false

G → ( H
  | λ

H → ArgList )
  | )

ArgList → Expr ArgList'

ArgList' → , Expr ArgList'
         | λ

# 2. primeros

## 2. Conjuntos Primeros

| No terminal | Primeros |
|---|---|
| `Program` | `{ int, float, bool, void, string }` |
| `DeclList` | `{ int, float, bool, void, string }` |
| `DeclList'` | `{ int, float, bool, void, string, λ }` |
| `Decl` | `{ int, float, bool, void, string }` |
| `C` | `{ (, ;, = }` |
| `A` | `{ ;, = }` |
| `VarDecl` | `{ int, float, bool, void, string }` |
| `FuncDecl` | `{ int, float, bool, void, string }` |
| `B` | `{ int, float, bool, void, string, ) }` |
| `Type` | `{ int, float, bool, void, string }` |
| `ParamList` | `{ int, float, bool, void, string }` |
| `ParamList'` | `{ ,, λ }` |
| `Param` | `{ int, float, bool, void, string }` |
| `Block` | `{ { }` |
| `StmtList` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, λ }` |
| `StmtList'` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, λ }` |
| `Stmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read }` |
| `I` | `{ =, ( }` |
| `K` | `{ !, -, (, id, num, str, true, false, ) }` |
| `J` | `{ !, -, (, id, num, str, true, false, ; }` |
| `IfStmt` | `{ if }` |
| `D` | `{ else, λ }` |
| `WhileStmt` | `{ while }` |
| `ForStmt` | `{ for }` |
| `Expr` | `{ !, -, (, id, num, str, true, false }` |
| `Expr'` | `{ \|\|, &&, λ }` |
| `RelExpr` | `{ !, -, (, id, num, str, true, false }` |
| `E` | `{ ==, !=, <, >, λ }` |
| `RelOp` | `{ ==, !=, <, > }` |
| `F` | `{ =, λ }` |
| `AddExpr` | `{ !, -, (, id, num, str, true, false }` |
| `AddExpr'` | `{ +, -, λ }` |
| `Term` | `{ !, -, (, id, num, str, true, false }` |
| `Term'` | `{ *, /, λ }` |
| `Unary` | `{ !, -, (, id, num, str, true, false }` |
| `Factor` | `{ (, id, num, str, true, false }` |
| `G` | `{ (, λ }` |
| `H` | `{ !, -, (, id, num, str, true, false, ) }` |
| `ArgList` | `{ !, -, (, id, num, str, true, false }` |
| `ArgList'` | `{ ,, λ }` |

3. Siguientes
Para calcular los "Siguientes" usamos las siguientes reglas:

Regla 1
Si A es el símbolo inicial, entonces:
# ∈ FOLLOW(A)

Regla 2
Si existe una producción:
B → α A β
entonces:
Primeros(β) - {λ} ⊆ Siguientes(A)

Regla 3
Si existe una producción:
B → α A
o:
B → α A β
donde:
β ⇒* λ
entonces:
Siguientes(B) ⊆ Siguientes(A)



## 3. Conjuntos Siguientes

| No terminal | Siguientes |
|---|---|
| `Program` | `{ # }` |
| `DeclList` | `{ # }` |
| `DeclList'` | `{ # }` |
| `Decl` | `{ int, float, bool, void, string, # }` |
| `Type` | `{ id }` |
| `C` | `{ int, float, bool, void, string, # }` |
| `A` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, }, # }` |
| `B` | `{ int, float, bool, void, string, # }` |
| `VarDecl` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `FuncDecl` | `∅` |
| `ParamList` | `{ ) }` |
| `ParamList'` | `{ ) }` |
| `Param` | `{ ,, ) }` |
| `Block` | `{ else, int, float, bool, void, string, id, if, while, for, return, print, read, }, # }` |
| `StmtList` | `{ } }` |
| `StmtList'` | `{ } }` |
| `Stmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `I` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `K` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `J` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `IfStmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `D` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `WhileStmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `ForStmt` | `{ int, float, bool, void, string, id, if, while, for, return, print, read, } }` |
| `Expr` | `{ ), ,, ; }` |
| `Expr'` | `{ ), ,, ; }` |
| `RelExpr` | `{ ||, &&, ), ,, ; }` |
| `E` | `{ ||, &&, ), ,, ; }` |
| `RelOp` | `{ !, -, (, id, num, str, true, false }` |
| `F` | `{ !, -, (, id, num, str, true, false }` |
| `AddExpr` | `{ ==, !=, <, >, ||, &&, ), ,, ; }` |
| `AddExpr'` | `{ ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Term` | `{ +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Term'` | `{ +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Unary` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `Factor` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `G` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `H` | `{ *, /, +, -, ==, !=, <, >, ||, &&, ), ,, ; }` |
| `ArgList` | `{ ) }` |
| `ArgList'` | `{ ) }` |